# Train & evaluate on Colab / Kaggle (thin wrapper)

Everything below calls the `captioner` package through `make`; no pipeline code lives in this notebook.

1. Runtime → Change runtime type → **T4 GPU**.
2. Run the cells in order. Full M2 training on cached features targets < 2 h on a T4 (NFR-3).
3. Copy `reports/<run>/metrics.json`, `predictions.csv` and the figures back into the repo; never commit data, features or checkpoints.

In [ ]:
REPO = 'https://github.com/Kislay01/DeepLearning_CP.git'
CONFIG = 'configs/attention.yaml'   # or configs/merge_baseline.yaml

In [ ]:
!git clone --depth 1 $REPO captioner_repo
%cd captioner_repo
!pip install -q uv
!java -version  # METEOR + PTB tokenizer need Java (preinstalled on Colab)

In [ ]:
!uv sync --frozen

## Pipeline
`make data` downloads Flickr8k (MD5-verified), builds the manifest and the train-only vocab; `make features` caches ResNet-50 features (≈1.6 GB HDF5); `make train` trains with early stopping on val BLEU-4; `make eval` scores the test split with beam 1/3/5.

In [ ]:
!make data CONFIG=$CONFIG
!uv run python scripts/verify_data.py --config $CONFIG --skip-md5

In [ ]:
!make features CONFIG=$CONFIG

In [ ]:
!make train CONFIG=$CONFIG

In [ ]:
!make eval CONFIG=$CONFIG

In [ ]:
!make export CONFIG=$CONFIG
!ls -la reports/*/ artifacts/release/*/

## Optional
- 3 seeds for E1/E4: `!make train CONFIG=$CONFIG` with `--set seed=1` / `seed=2` via `uv run captioner train -c $CONFIG --set seed=1`.
- E5 fine-tuning: `!uv run captioner train -c configs/attention_finetune.yaml --set train.init_from=artifacts/runs/<run>/best.pt`.
- TensorBoard: `%load_ext tensorboard` then `%tensorboard --logdir artifacts/runs`.